# NB_patient_360 — Silver to Gold

Builds `adb_caresync.gold.patient_360` (one row per patient) per the STTM `patient_360` sheet: patients LEFT JOIN hospitals, insurance_providers and aggregated lab_results. Full rebuild (CREATE OR REPLACE TABLE AS SELECT).

Note: silver `hospitals` and `lab_results` currently contain duplicate key rows, so every source is de-duplicated on its business key (latest `update_timestamp`) to preserve the one-row-per-patient grain.

In [0]:
%sql
-- Gold: patient_360 | Grain: one row per patient | Load: FULL rebuild
CREATE OR REPLACE TABLE adb_caresync.gold.patient_360
COMMENT 'Patient 360 view: demographics, registered hospital, insurance provider and lab test summary. One row per patient.'
AS
WITH patients AS (
  SELECT * FROM adb_caresync.silver.patients
  QUALIFY row_number() OVER (PARTITION BY patient_id ORDER BY update_timestamp DESC) = 1
),
hospitals AS (
  SELECT hospital_id, hospital_name FROM adb_caresync.silver.hospitals
  QUALIFY row_number() OVER (PARTITION BY hospital_id ORDER BY update_timestamp DESC, landing_timestamp DESC) = 1
),
insurance_providers AS (
  SELECT insurance_provider_id, provider_name FROM adb_caresync.silver.insurance_providers
  QUALIFY row_number() OVER (PARTITION BY insurance_provider_id ORDER BY update_timestamp DESC) = 1
),
lab_results AS (
  SELECT * FROM adb_caresync.silver.lab_results
  QUALIFY row_number() OVER (PARTITION BY lab_result_id ORDER BY update_timestamp DESC) = 1
),
lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                            AS total_tests,
    COUNT_IF(result_status = 'Abnormal')                AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')                AS critical_test_count,
    MAX(test_date)                                      AS last_test_date
  FROM lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                              AS patient_id,
  p.first_name                                                           AS first_name,
  p.last_name                                                            AS last_name,
  CAST(p.date_of_birth AS DATE)                                          AS date_of_birth,
  CAST(floor(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender                                                               AS gender,
  p.city                                                                 AS city,
  p.state                                                                AS state,
  CAST(p.registered_hospital_id AS INT)                                  AS hospital_id,
  h.hospital_name                                                        AS hospital_name,
  CAST(p.insurance_provider_id AS INT)                                   AS insurance_provider_id,
  i.provider_name                                                        AS insurance_provider_name,
  CAST(coalesce(l.total_tests, 0) AS INT)                                AS total_tests,
  CAST(coalesce(l.abnormal_test_count, 0) AS INT)                        AS abnormal_test_count,
  CAST(coalesce(l.critical_test_count, 0) AS INT)                        AS critical_test_count,
  l.last_test_date                                                       AS last_test_date
FROM patients p
LEFT JOIN hospitals h            ON p.registered_hospital_id = h.hospital_id
LEFT JOIN insurance_providers i  ON p.insurance_provider_id = i.insurance_provider_id
LEFT JOIN lab_agg l              ON p.patient_id = l.patient_id;

In [0]:
%python
# Return the number of records written to gold.patient_360 (full rebuild = total rows in table)
# ADF: read via @activity('<NotebookActivity>').output.runOutput
target_table = "adb_caresync.gold.patient_360"
record_count = spark.table(target_table).count()
print(f"FULL load completed for {target_table}. record_count = {record_count}")
dbutils.notebook.exit(str(record_count))